# Constructive heuristics (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/constructive-heuristics_python.ipynb)

Executable notebook of the chapter “Constructive heuristics” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the constructive heuristics of the chapter for the knapsack problem, the bin packing problem, $P\,||\,C_{\max}$ and the TSP, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/constructive-heuristics/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/constructive-heuristics_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. Nothing to install: only the Python standard library is used.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

Constructive heuristics for the KP, the BPP, P||Cmax and the TSP: items, bins, jobs,
machines and cities are indexed starting from 0, and ties are always broken in favour
of the smallest index.

In Python the functions **return** their results instead of filling output arrays:
for example `greedy_order_kp` returns the pair `(z, x)`, the TSP functions return `(L, tour)`.
The seed of the random generator is an integer that is returned updated
(`park_miller` returns `(u, seed)`, `randomized_nn` returns `(L, tour, seed)`),
instead of being modified through a pointer.

## Sorting

The orders of the KP items and the decreasing order used by FFD, BFD and $LPT$ are computed by insertion sort, which keeps the index order among equal keys. The efficiencies are compared without divisions: $p_i/w_i > p_j/w_j$ if and only if $p_i w_j > p_j w_i$.

In [1]:
def before(i, j, p, w, criterion):
    """True if item i comes before item j for the criterion (0 profit, 1 weight, 2 efficiency)."""
    if criterion == 0:
        return p[i] > p[j]
    if criterion == 1:
        return w[i] < w[j]
    return p[i] * w[j] > p[j] * w[i]       # p_i/w_i > p_j/w_j without divisions


def kp_order(n, p, w, criterion):
    """Returns the items sorted by the criterion:
    0 = non-increasing profit, 1 = non-decreasing weight, 2 = non-increasing efficiency p/w.
    Insertion sort: items with equal keys keep their index order."""
    order = [0] * n
    for k in range(n):
        i = k                              # item to be inserted in order[0..k-1]
        pos = k
        while pos > 0 and before(i, order[pos - 1], p, w, criterion):
            order[pos] = order[pos - 1]
            pos -= 1
        order[pos] = i
    return order

In [2]:
def decreasing_order(n, a):
    """Returns the indices sorted by non-increasing a[i].
    Insertion sort: indices with equal values keep their order."""
    order = [0] * n
    for k in range(n):
        pos = k
        while pos > 0 and a[k] > a[order[pos - 1]]:
            order[pos] = order[pos - 1]
            pos -= 1
        order[pos] = k
    return order

## The knapsack problem

The greedy algorithm with a given order of the items.

In [3]:
def greedy_order_kp(n, c, w, p, order):
    """The items are examined in the order given by order[0..n-1].
    Returns (z, x): the profit z of the solution and x[i] = 1 if item i is selected."""
    x = [0] * n                            # every item is initially not selected
    rc = c                                 # residual capacity
    z = 0                                  # profit of the selected items

    for k in range(n):
        i = order[k]                       # k-th item in the order
        if w[i] <= rc:                     # feasibility test
            x[i] = 1                       # select item i
            rc = rc - w[i]                 # update residual capacity
            z = z + p[i]
    return z, x

## The bin packing problem

First Fit Decreasing and Best Fit Decreasing.

In [4]:
def first_fit_decreasing(n, c, w):
    """The items are sorted by non-increasing weight and packed by First Fit.
    Returns (nb, bin_): the number of bins used and bin_[i] = bin assigned to item i."""
    order = decreasing_order(n, w)         # sort: largest items first

    bin_ = [0] * n
    rc = [0] * n                           # residual capacities (at most n bins)
    nb = 0                                 # number of bins opened so far

    for k in range(n):
        i = order[k]                       # k-th largest item
        s = -1                             # selected bin (-1 means: not found yet)
        for b in range(nb):                # first bin in which item i fits
            if w[i] <= rc[b]:
                s = b
                break
        if s == -1:                        # no bin fits: open a new one
            rc[nb] = c
            s = nb
            nb += 1
        bin_[i] = s                        # pack item i into bin s
        rc[s] = rc[s] - w[i]
    return nb, bin_

In [5]:
def best_fit_decreasing(n, c, w):
    """The items are sorted by non-increasing weight and packed by Best Fit.
    Returns (nb, bin_): the number of bins used and bin_[i] = bin assigned to item i."""
    order = decreasing_order(n, w)         # sort: largest items first

    bin_ = [0] * n
    rc = [0] * n                           # residual capacities (at most n bins)
    nb = 0                                 # number of bins opened so far

    for k in range(n):
        i = order[k]                       # k-th largest item
        s = -1                             # selected bin (-1 means: not found yet)
        for b in range(nb):                # fullest bin in which item i fits
            if w[i] <= rc[b] and (s == -1 or rc[b] < rc[s]):
                s = b
        if s == -1:                        # no bin fits: open a new one
            rc[nb] = c
            s = nb
            nb += 1
        bin_[i] = s                        # pack item i into bin s
        rc[s] = rc[s] - w[i]
    return nb, bin_

## Scheduling on parallel machines

List Scheduling and Longest Processing Time first ($LPT$).

In [6]:
def list_scheduling(n, m, t, order):
    """The jobs are examined in the order given by order[0..n-1];
    each job is assigned to the machine with the smallest current load.
    Returns (Cmax, mach): the makespan and mach[j] = machine of job j."""
    mach = [0] * n
    L = [0] * m                            # loads of the machines

    for k in range(n):
        j = order[k]                       # k-th job of the list
        s = 0                              # least loaded machine
        for h in range(1, m):
            if L[h] < L[s]:
                s = h
        mach[j] = s                        # assign job j to machine s
        L[s] = L[s] + t[j]

    Cmax = 0                               # makespan = largest load
    for h in range(m):
        if L[h] > Cmax:
            Cmax = L[h]
    return Cmax, mach

In [7]:
def lpt(n, m, t):
    """Longest Processing Time first: list scheduling with the jobs
    sorted by non-increasing processing time.
    Returns (Cmax, mach): the makespan and mach[j] = machine of job j."""
    order = decreasing_order(n, t)         # longest jobs first
    return list_scheduling(n, m, t, order)

## The travelling salesman problem

The distances are stored in an $n\times n$ matrix `d`, and a tour in an array of $n$ cities. In `Insertion` the three selection rules share the same loop: the value of a candidate is its distance from the subtour ($NI$), the same distance with the sign changed, so that the minimum corresponds to the farthest city ($FI$), or its minimum insertion cost ($CI$).

In [8]:
def tour_length(n, d, tour):
    """Length of the tour tour[0], tour[1], ..., tour[n-1], tour[0]."""
    L = 0
    for k in range(n - 1):
        L = L + d[tour[k]][tour[k + 1]]
    return L + d[tour[n - 1]][tour[0]]     # closing edge

In [9]:
def nearest_neighbour(n, d, s):
    """Starting from city s, the tour is extended at its end with the
    nearest city not yet visited. Returns (L, tour): the length and the tour."""
    visited = [False] * n
    tour = [0] * n
    tour[0] = s                            # the tour starts at city s
    visited[s] = True

    for length in range(1, n):
        last = tour[length - 1]            # last city of the partial tour
        k = -1                             # nearest unvisited city
        for j in range(n):
            if not visited[j] and (k == -1 or d[last][j] < d[last][k]):
                k = j
        tour[length] = k                   # append k at the end
        visited[k] = True

    return tour_length(n, d, tour), tour

In [10]:
def multi_start_nn(n, d):
    """Runs the nearest neighbour heuristic from every city and keeps the shortest
    tour (the first one found, in case of ties). Returns (L, tour)."""
    tour = [0] * n
    best = -1                              # length of the best tour found
    for s in range(n):
        L, cur = nearest_neighbour(n, d, s)
        if best == -1 or L < best:
            best = L
            for k in range(n):
                tour[k] = cur[k]
    return best, tour

In [11]:
def best_position(length, d, tour, k):
    """Cheapest position for city k in the subtour tour[0..length-1]:
    returns (increase of length, a), where (tour[a], tour[a+1]) is the edge to break."""
    best = -1
    pos = 0
    for a in range(length):
        i = tour[a]
        j = tour[(a + 1) % length]
        cost = d[i][k] + d[k][j] - d[i][j]
        if best == -1 or cost < best:
            best = cost
            pos = a
    return best, pos


def insertion(n, d, s, rule):
    """Insertion heuristics: starting from the subtour (s), the city selected by the rule
    is inserted into the subtour where it increases the length the least.
    rule: 0 = nearest insertion (NI), 1 = farthest insertion (FI), 2 = cheapest insertion (CI).
    Returns (L, tour)."""
    inside = [False] * n                   # inside[i] = True if city i is in the subtour
    tour = [0] * n
    tour[0] = s                            # the subtour (s)
    inside[s] = True

    for length in range(1, n):
        # selection step: city k to be inserted
        k = -1
        keyk = 0
        for q in range(n):
            if inside[q]:
                continue
            if rule == 2:
                key, a = best_position(length, d, tour, q)   # insertion cost
            else:
                key = d[q][tour[0]]        # distance from the subtour
                for a in range(1, length):
                    if d[q][tour[a]] < key:
                        key = d[q][tour[a]]
                if rule == 1:
                    key = -key             # farthest: maximize the distance
            if k == -1 or key < keyk:
                k = q
                keyk = key
        # insertion step: k goes where it increases the length the least
        cost, pos = best_position(length, d, tour, k)
        for a in range(length, pos + 1, -1):
            tour[a] = tour[a - 1]
        tour[pos + 1] = k
        inside[k] = True

    return tour_length(n, d, tour), tour

## Randomized construction

The Park–Miller generator and the randomized nearest neighbour. The threshold of the RCL is tested in integer arithmetic, with $\alpha$ given as a fraction $\alpha_{num}/\alpha_{den}$, so that no rounding error can change the list.

In [12]:
def park_miller(seed):
    """Park-Miller minimal standard generator:
    seed <- 16807 * seed mod (2^31 - 1), and u = seed / (2^31 - 1) in (0,1).
    Returns (u, new seed)."""
    seed = (16807 * seed) % 2147483647
    return seed / 2147483647.0, seed

In [13]:
def randomized_nn(n, d, s, alpha_num, alpha_den, seed):
    """Randomized nearest neighbour with a value-based restricted candidate list:
    RCL = { j unvisited : d[last][j] <= dmin + alpha (dmax - dmin) },
    with alpha = alpha_num / alpha_den; the next city is drawn uniformly from the RCL
    (a random number is used only when the RCL has at least two cities).
    Returns (L, tour, seed), with the updated seed."""
    visited = [False] * n
    RCL = [0] * n
    tour = [0] * n
    tour[0] = s
    visited[s] = True

    for length in range(1, n):
        last = tour[length - 1]
        dmin = -1                          # smallest distance from last
        dmax = -1                          # largest distance from last
        for j in range(n):
            if visited[j]:
                continue
            if dmin == -1 or d[last][j] < dmin:
                dmin = d[last][j]
            if dmax == -1 or d[last][j] > dmax:
                dmax = d[last][j]
        size = 0                           # restricted candidate list
        for j in range(n):
            # d <= dmin + alpha (dmax - dmin), in integer arithmetic
            if not visited[j] and alpha_den * d[last][j] <= alpha_den * dmin + alpha_num * (dmax - dmin):
                RCL[size] = j
                size += 1
        k = RCL[0]
        if size > 1:
            u, seed = park_miller(seed)    # random number in (0,1)
            k = RCL[int(u * size)]         # uniform choice in the RCL
        tour[length] = k
        visited[k] = True

    return tour_length(n, d, tour), tour, seed

## Regret and look-ahead

Regret-2 insertion and the pilot method with the nearest neighbour.

In [14]:
def regret2_insertion(n, d, s):
    """Regret-2 insertion: starting from the subtour (s), insert the city with the largest
    regret = (second cheapest insertion cost) - (cheapest insertion cost),
    ties broken by the smaller cheapest cost, then by the smaller index,
    at its cheapest position. Returns (L, tour)."""
    inside = [False] * n
    tour = [0] * n
    tour[0] = s
    inside[s] = True

    for length in range(1, n):
        k = -1
        regk = 0
        c1k = 0
        posk = 0
        for q in range(n):
            if inside[q]:
                continue
            c1 = -1                        # cheapest cost
            c2 = -1                        # second cheapest cost
            pos = 0
            for a in range(length):
                i = tour[a]
                j = tour[(a + 1) % length]
                cost = d[i][q] + d[q][j] - d[i][j]
                if c1 == -1 or cost < c1:
                    c2 = c1
                    c1 = cost
                    pos = a
                elif c2 == -1 or cost < c2:
                    c2 = cost
            if c2 == -1:
                c2 = c1                    # a single position: no regret
            reg = c2 - c1
            if k == -1 or reg > regk or (reg == regk and c1 < c1k):
                k = q
                regk = reg
                c1k = c1
                posk = pos
        for a in range(length, posk + 1, -1):   # insert k after position posk
            tour[a] = tour[a - 1]
        tour[posk + 1] = k
        inside[k] = True

    return tour_length(n, d, tour), tour

In [15]:
def pilot_nn(n, d, s):
    """Pilot method with the nearest neighbour heuristic as pilot:
    every candidate next city is evaluated by completing the partial tour
    with nearest neighbour; the candidate with the shortest completed tour is appended.
    Returns (L, tour)."""
    visited = [False] * n
    trial = [0] * n                        # partial tour completed by the pilot
    used = [False] * n
    tour = [0] * n
    tour[0] = s
    visited[s] = True

    for length in range(1, n):
        k = -1
        bestL = -1
        for q in range(n):                 # candidate next city q
            if visited[q]:
                continue
            for i in range(n):
                used[i] = visited[i]
            for a in range(length):
                trial[a] = tour[a]
            trial[length] = q
            used[q] = True
            for t in range(length + 1, n): # pilot: nearest neighbour completion
                last = trial[t - 1]
                nxt = -1
                for j in range(n):
                    if not used[j] and (nxt == -1 or d[last][j] < d[last][nxt]):
                        nxt = j
                trial[t] = nxt
                used[nxt] = True
            L = tour_length(n, d, trial)
            if k == -1 or L < bestL:
                k = q
                bestL = L
        tour[length] = k                   # append the best candidate
        visited[k] = True

    return tour_length(n, d, tour), tour

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [16]:
# runs every example of the lecture notes and every instance of the exercises
errors = 0
checks = 0


def check(instance, algorithm, found, expected):
    global checks, errors
    checks += 1
    if found != expected:
        errors += 1
    print("%-18s %-18s %4d %s" % (instance, algorithm, found,
          "OK" if found == expected else "<-- EXPECTED A DIFFERENT VALUE"))


def check_vec(instance, what, found, expected, n):
    global checks, errors
    checks += 1
    ok = True
    for k in range(n):
        if found[k] != expected[k]:
            ok = False
    if not ok:
        errors += 1
    line = "%-18s %-18s " % (instance, what)
    for k in range(n):
        line += "%d " % (found[k] + 1)
    print(line + ("OK" if ok else "<-- EXPECTED A DIFFERENT SOLUTION"))


e1 = [1, 1, 0, 0]
e2 = [0, 0, 1, 1]
e3 = [1, 0, 1, 0]
e4 = [0, 1, 0, 0, 0]
e5 = [1, 0, 0, 1, 0]
e6 = [0, 0, 1, 1, 0]
e7 = [1, 0, 0, 0, 1]
e8 = [0, 1, 0, 1, 1]
e9 = [1, 1, 0, 0, 0]
e10 = [0, 1, 0, 0, 0]
e11 = [0, 0, 1, 0, 1]
e12 = [1, 0, 0, 0, 1]
e13 = [1, 2, 0, 1, 0, 2]
e14 = [1, 2, 0, 1, 0, 2]
e15 = [1, 1, 0, 1, 2, 0]
e16 = [1, 1, 0, 1, 2, 0]
e17 = [0, 1, 2, 2, 0, 2, 1]
e18 = [0, 1, 2, 2, 0, 2, 1]
e19 = [2, 0, 1, 2, 0, 2, 1, 3]
e20 = [2, 0, 1, 2, 0, 2, 1, 3]
e21 = [1, 2, 1, 0, 0, 2, 2]
e22 = [1, 2, 1, 0, 2, 0, 2]
e23 = [0, 1, 2, 2, 0, 1, 0]
e24 = [0, 2, 1, 0, 2, 1, 0]
e25 = [0, 1, 2, 2, 0, 1, 0]
e26 = [0, 1, 2, 0, 2, 0, 1, 0]
e27 = [2, 1, 1, 2, 2, 0, 0, 0]
e28 = [0, 1, 2, 0, 2, 1, 0]
e29 = [1, 2, 2, 1, 0, 0, 1]
e30 = [0, 1, 2, 0, 1, 2, 0]
e31 = [2, 0, 1, 1, 0, 2, 1]
e32 = [0, 2, 1, 3, 4]
e33 = [1, 3, 0, 2, 4]
e34 = [2, 0, 3, 1, 4]
e35 = [3, 0, 2, 1, 4]
e36 = [4, 1, 3, 0, 2]
e37 = [1, 3, 0, 2, 4]
e38 = [0, 3, 1, 4, 2]
e39 = [0, 3, 1, 4, 2]
e40 = [0, 3, 1, 4, 2]
e41 = [1, 4, 2, 0, 3]
e42 = [1, 4, 2, 0, 3]
e43 = [1, 4, 2, 0, 3]
e44 = [2, 4, 1, 3, 0]
e45 = [2, 0, 3, 1, 4]
e46 = [2, 4, 1, 3, 0]
e47 = [3, 1, 4, 2, 0]
e48 = [3, 0, 2, 4, 1]
e49 = [3, 1, 4, 2, 0]
e50 = [4, 2, 0, 3, 1]
e51 = [4, 1, 3, 0, 2]
e52 = [4, 2, 0, 3, 1]
e53 = [0, 3, 1, 4, 2]
e54 = [0, 4, 1, 3, 2]
e55 = [0, 2, 1, 3, 4]
e56 = [0, 3, 1, 4, 2]
e57 = [0, 2, 1, 3, 4]
e58 = [0, 3, 2, 1, 4]
e59 = [0, 5, 2, 1, 3, 4]
e60 = [1, 2, 5, 0, 3, 4]
e61 = [2, 1, 5, 0, 3, 4]
e62 = [3, 0, 5, 2, 1, 4]
e63 = [4, 5, 0, 2, 1, 3]
e64 = [5, 0, 2, 1, 3, 4]
e65 = [1, 2, 5, 0, 3, 4]
e66 = [0, 4, 1, 2, 3, 5]
e67 = [0, 5, 4, 1, 2, 3]
e68 = [0, 3, 1, 2, 4, 5]
e69 = [0, 3, 2, 1, 4, 5]
e70 = [0, 3, 1, 2, 4, 5]
e71 = [0, 4, 2, 1, 5, 3]
e72 = [1, 2, 5, 3, 4, 0]
e73 = [2, 1, 5, 3, 4, 0]
e74 = [3, 5, 2, 1, 4, 0]
e75 = [4, 0, 1, 2, 5, 3]
e76 = [5, 3, 2, 1, 4, 0]
e77 = [1, 2, 5, 3, 4, 0]
e78 = [0, 1, 2, 5, 3, 4]
e79 = [0, 1, 2, 5, 3, 4]
e80 = [0, 1, 2, 5, 3, 4]
e81 = [0, 4, 1, 5, 3, 2]
e82 = [1, 5, 0, 4, 2, 3]
e83 = [2, 0, 4, 1, 5, 3]
e84 = [3, 0, 4, 1, 5, 2]
e85 = [4, 0, 1, 5, 3, 2]
e86 = [5, 1, 0, 4, 2, 3]
e87 = [4, 0, 1, 5, 3, 2]
e88 = [0, 5, 1, 4, 2, 3]
e89 = [0, 1, 5, 3, 2, 4]
e90 = [0, 2, 3, 5, 1, 4]
e91 = [0, 1, 3, 2, 4]
e92 = [1, 0, 3, 2, 4]
e93 = [2, 3, 0, 1, 4]
e94 = [3, 2, 4, 1, 0]
e95 = [4, 2, 3, 0, 1]
e96 = [1, 0, 3, 2, 4]
e97 = [0, 4, 2, 3, 1]
e98 = [0, 1, 4, 2, 3]
e99 = [0, 4, 2, 3, 1]
e100 = [0, 1, 2, 3, 4]
e101 = [0, 3, 2, 4, 1]

p = [10, 7, 6, 4]
w = [5, 4, 3, 3]
order = kp_order(4, p, w, 0)
z, x = greedy_order_kp(4, 9, w, p, order)
check("KP reference", "greedy profit", z, 17)
check_vec("KP reference", "  x", x, e1, 4)
order = kp_order(4, p, w, 1)
z, x = greedy_order_kp(4, 9, w, p, order)
check("KP reference", "greedy weight", z, 10)
check_vec("KP reference", "  x", x, e2, 4)
order = kp_order(4, p, w, 2)
z, x = greedy_order_kp(4, 9, w, p, order)
check("KP reference", "greedy efficiency", z, 16)
check_vec("KP reference", "  x", x, e3, 4)

p = [4, 15, 11, 8, 2]
w = [3, 8, 7, 2, 5]
order = kp_order(5, p, w, 0)
z, x = greedy_order_kp(5, 9, w, p, order)
check("EX_KO_1", "greedy profit", z, 15)
check_vec("EX_KO_1", "  x", x, e4, 5)
order = kp_order(5, p, w, 1)
z, x = greedy_order_kp(5, 9, w, p, order)
check("EX_KO_1", "greedy weight", z, 12)
check_vec("EX_KO_1", "  x", x, e5, 5)
order = kp_order(5, p, w, 2)
z, x = greedy_order_kp(5, 9, w, p, order)
check("EX_KO_1", "greedy efficiency", z, 19)
check_vec("EX_KO_1", "  x", x, e6, 5)

p = [9, 4, 8, 6, 7]
w = [7, 2, 9, 5, 6]
order = kp_order(5, p, w, 0)
z, x = greedy_order_kp(5, 13, w, p, order)
check("EX_KO_2", "greedy profit", z, 16)
check_vec("EX_KO_2", "  x", x, e7, 5)
order = kp_order(5, p, w, 1)
z, x = greedy_order_kp(5, 13, w, p, order)
check("EX_KO_2", "greedy weight", z, 17)
check_vec("EX_KO_2", "  x", x, e8, 5)
order = kp_order(5, p, w, 2)
z, x = greedy_order_kp(5, 13, w, p, order)
check("EX_KO_2", "greedy efficiency", z, 13)
check_vec("EX_KO_2", "  x", x, e9, 5)

p = [8, 13, 5, 12, 7]
w = [5, 9, 4, 8, 2]
order = kp_order(5, p, w, 0)
z, x = greedy_order_kp(5, 10, w, p, order)
check("EX_KO_3", "greedy profit", z, 13)
check_vec("EX_KO_3", "  x", x, e10, 5)
order = kp_order(5, p, w, 1)
z, x = greedy_order_kp(5, 10, w, p, order)
check("EX_KO_3", "greedy weight", z, 12)
check_vec("EX_KO_3", "  x", x, e11, 5)
order = kp_order(5, p, w, 2)
z, x = greedy_order_kp(5, 10, w, p, order)
check("EX_KO_3", "greedy efficiency", z, 15)
check_vec("EX_KO_3", "  x", x, e12, 5)

w = [4, 4, 5, 3, 2, 3]
nb, bin_ = first_fit_decreasing(6, 7, w)
check("BPP reference", "FFD", nb, 3)
check_vec("BPP reference", "  bin", bin_, e13, 6)
nb, bin_ = best_fit_decreasing(6, 7, w)
check("BPP reference", "BFD", nb, 3)
check_vec("BPP reference", "  bin", bin_, e14, 6)

w = [2, 2, 3, 2, 2, 3]
nb, bin_ = first_fit_decreasing(6, 7, w)
check("BPP FF<FFD", "FFD", nb, 3)
check_vec("BPP FF<FFD", "  bin", bin_, e15, 6)
nb, bin_ = best_fit_decreasing(6, 7, w)
check("BPP FF<FFD", "BFD", nb, 3)
check_vec("BPP FF<FFD", "  bin", bin_, e16, 6)

w = [3, 3, 5, 2, 6, 2, 6]
nb, bin_ = first_fit_decreasing(7, 10, w)
check("EX_BD_1", "FFD", nb, 3)
check_vec("EX_BD_1", "  bin", bin_, e17, 7)
nb, bin_ = best_fit_decreasing(7, 10, w)
check("EX_BD_1", "BFD", nb, 3)
check_vec("EX_BD_1", "  bin", bin_, e18, 7)

w = [2, 6, 3, 5, 4, 2, 6, 2]
nb, bin_ = first_fit_decreasing(8, 10, w)
check("EX_BD_2", "FFD", nb, 4)
check_vec("EX_BD_2", "  bin", bin_, e19, 8)
nb, bin_ = best_fit_decreasing(8, 10, w)
check("EX_BD_2", "BFD", nb, 4)
check_vec("EX_BD_2", "  bin", bin_, e20, 8)

w = [4, 4, 5, 7, 2, 2, 4]
nb, bin_ = first_fit_decreasing(7, 10, w)
check("EX_BD_3", "FFD", nb, 3)
check_vec("EX_BD_3", "  bin", bin_, e21, 7)
nb, bin_ = best_fit_decreasing(7, 10, w)
check("EX_BD_3", "BFD", nb, 3)
check_vec("EX_BD_3", "  bin", bin_, e22, 7)

t = [5, 5, 4, 4, 3, 3, 3]
o = [0, 1, 2, 3, 4, 5, 6]
cmax, mach = list_scheduling(7, 3, t, o)
check("P||Cmax reference", "LIST input", cmax, 11)
check_vec("P||Cmax reference", "  mach", mach, e23, 7)
o = [6, 5, 4, 3, 2, 1, 0]
cmax, mach = list_scheduling(7, 3, t, o)
check("P||Cmax reference", "LIST reverse", cmax, 12)
check_vec("P||Cmax reference", "  mach", mach, e24, 7)
cmax, mach = lpt(7, 3, t)
check("P||Cmax reference", "LPT", cmax, 11)
check_vec("P||Cmax reference", "  mach", mach, e25, 7)

t = [2, 8, 7, 6, 8, 3, 4, 9]
o = [0, 1, 2, 3, 4, 5, 6, 7]
cmax, mach = list_scheduling(8, 3, t, o)
check("EX_LPT_1", "LIST input", cmax, 20)
check_vec("EX_LPT_1", "  mach", mach, e26, 8)
cmax, mach = lpt(8, 3, t)
check("EX_LPT_1", "LPT", cmax, 16)
check_vec("EX_LPT_1", "  mach", mach, e27, 8)

t = [3, 6, 5, 4, 4, 9, 7]
o = [0, 1, 2, 3, 4, 5, 6]
cmax, mach = list_scheduling(7, 3, t, o)
check("EX_LPT_2", "LIST input", cmax, 15)
check_vec("EX_LPT_2", "  mach", mach, e28, 7)
cmax, mach = lpt(7, 3, t)
check("EX_LPT_2", "LPT", cmax, 14)
check_vec("EX_LPT_2", "  mach", mach, e29, 7)

t = [5, 8, 8, 4, 5, 7, 4]
o = [0, 1, 2, 3, 4, 5, 6]
cmax, mach = list_scheduling(7, 3, t, o)
check("EX_LPT_3", "LIST input", cmax, 15)
check_vec("EX_LPT_3", "  mach", mach, e30, 7)
cmax, mach = lpt(7, 3, t)
check("EX_LPT_3", "LPT", cmax, 16)
check_vec("EX_LPT_3", "  mach", mach, e31, 7)

d = [[0, 5, 2, 2, 9], [5, 0, 4, 3, 4], [2, 4, 0, 4, 7], [2, 3, 4, 0, 7], [9, 4, 7, 7, 0]]
L, tour = nearest_neighbour(5, d, 0)
check("TSP reference", "NN from 1", L, 25)
check_vec("TSP reference", "  tour", tour, e32, 5)
L, tour = nearest_neighbour(5, d, 1)
check("TSP reference", "NN from 2", L, 18)
check_vec("TSP reference", "  tour", tour, e33, 5)
L, tour = nearest_neighbour(5, d, 2)
check("TSP reference", "NN from 3", L, 18)
check_vec("TSP reference", "  tour", tour, e34, 5)
L, tour = nearest_neighbour(5, d, 3)
check("TSP reference", "NN from 4", L, 19)
check_vec("TSP reference", "  tour", tour, e35, 5)
L, tour = nearest_neighbour(5, d, 4)
check("TSP reference", "NN from 5", L, 18)
check_vec("TSP reference", "  tour", tour, e36, 5)
L, tour = multi_start_nn(5, d)
check("TSP reference", "multi-start NN", L, 18)
check_vec("TSP reference", "  tour", tour, e37, 5)
L, tour = insertion(5, d, 0, 0)
check("TSP reference", "NI from 1", L, 18)
check_vec("TSP reference", "  tour", tour, e38, 5)
L, tour = insertion(5, d, 0, 1)
check("TSP reference", "FI from 1", L, 18)
check_vec("TSP reference", "  tour", tour, e39, 5)
L, tour = insertion(5, d, 0, 2)
check("TSP reference", "CI from 1", L, 18)
check_vec("TSP reference", "  tour", tour, e40, 5)
L, tour = insertion(5, d, 1, 0)
check("TSP reference", "NI from 2", L, 18)
check_vec("TSP reference", "  tour", tour, e41, 5)
L, tour = insertion(5, d, 1, 1)
check("TSP reference", "FI from 2", L, 18)
check_vec("TSP reference", "  tour", tour, e42, 5)
L, tour = insertion(5, d, 1, 2)
check("TSP reference", "CI from 2", L, 18)
check_vec("TSP reference", "  tour", tour, e43, 5)
L, tour = insertion(5, d, 2, 0)
check("TSP reference", "NI from 3", L, 18)
check_vec("TSP reference", "  tour", tour, e44, 5)
L, tour = insertion(5, d, 2, 1)
check("TSP reference", "FI from 3", L, 18)
check_vec("TSP reference", "  tour", tour, e45, 5)
L, tour = insertion(5, d, 2, 2)
check("TSP reference", "CI from 3", L, 18)
check_vec("TSP reference", "  tour", tour, e46, 5)
L, tour = insertion(5, d, 3, 0)
check("TSP reference", "NI from 4", L, 18)
check_vec("TSP reference", "  tour", tour, e47, 5)
L, tour = insertion(5, d, 3, 1)
check("TSP reference", "FI from 4", L, 18)
check_vec("TSP reference", "  tour", tour, e48, 5)
L, tour = insertion(5, d, 3, 2)
check("TSP reference", "CI from 4", L, 18)
check_vec("TSP reference", "  tour", tour, e49, 5)
L, tour = insertion(5, d, 4, 0)
check("TSP reference", "NI from 5", L, 18)
check_vec("TSP reference", "  tour", tour, e50, 5)
L, tour = insertion(5, d, 4, 1)
check("TSP reference", "FI from 5", L, 18)
check_vec("TSP reference", "  tour", tour, e51, 5)
L, tour = insertion(5, d, 4, 2)
check("TSP reference", "CI from 5", L, 18)
check_vec("TSP reference", "  tour", tour, e52, 5)
L, tour = regret2_insertion(5, d, 0)
check("TSP reference", "regret-2 from 1", L, 18)
check_vec("TSP reference", "  tour", tour, e53, 5)
L, tour = pilot_nn(5, d, 0)
check("TSP reference", "pilot NN from 1", L, 22)
check_vec("TSP reference", "  tour", tour, e54, 5)
seed = 1
L, tour, seed = randomized_nn(5, d, 0, 3, 10, seed)
check("TSP reference", "RCL run 1", L, 25)
check_vec("TSP reference", "  tour", tour, e55, 5)
L, tour, seed = randomized_nn(5, d, 0, 3, 10, seed)
check("TSP reference", "RCL run 2", L, 18)
check_vec("TSP reference", "  tour", tour, e56, 5)
L, tour, seed = randomized_nn(5, d, 0, 3, 10, seed)
check("TSP reference", "RCL run 3", L, 25)
check_vec("TSP reference", "  tour", tour, e57, 5)
L, tour, seed = randomized_nn(5, d, 0, 3, 10, seed)
check("TSP reference", "RCL run 4", L, 23)
check_vec("TSP reference", "  tour", tour, e58, 5)

d = [[0, 5, 4, 4, 5, 2], [5, 0, 1, 5, 5, 4], [4, 1, 0, 4, 5, 3], [4, 5, 4, 0, 9, 6], [5, 5, 5, 9, 0, 3], [2, 4, 3, 6, 3, 0]]
L, tour = nearest_neighbour(6, d, 0)
check("TSP 6 cities", "NN from 1", L, 25)
check_vec("TSP 6 cities", "  tour", tour, e59, 6)
L, tour = nearest_neighbour(6, d, 1)
check("TSP 6 cities", "NN from 2", L, 24)
check_vec("TSP 6 cities", "  tour", tour, e60, 6)
L, tour = nearest_neighbour(6, d, 2)
check("TSP 6 cities", "NN from 3", L, 25)
check_vec("TSP 6 cities", "  tour", tour, e61, 6)
L, tour = nearest_neighbour(6, d, 3)
check("TSP 6 cities", "NN from 4", L, 24)
check_vec("TSP 6 cities", "  tour", tour, e62, 6)
L, tour = nearest_neighbour(6, d, 4)
check("TSP 6 cities", "NN from 5", L, 24)
check_vec("TSP 6 cities", "  tour", tour, e63, 6)
L, tour = nearest_neighbour(6, d, 5)
check("TSP 6 cities", "NN from 6", L, 24)
check_vec("TSP 6 cities", "  tour", tour, e64, 6)
L, tour = multi_start_nn(6, d)
check("TSP 6 cities", "multi-start NN", L, 24)
check_vec("TSP 6 cities", "  tour", tour, e65, 6)
L, tour = insertion(6, d, 0, 0)
check("TSP 6 cities", "NI from 1", L, 23)
check_vec("TSP 6 cities", "  tour", tour, e66, 6)
L, tour = insertion(6, d, 0, 1)
check("TSP 6 cities", "FI from 1", L, 19)
check_vec("TSP 6 cities", "  tour", tour, e67, 6)
L, tour = insertion(6, d, 0, 2)
check("TSP 6 cities", "CI from 1", L, 20)
check_vec("TSP 6 cities", "  tour", tour, e68, 6)
L, tour = regret2_insertion(6, d, 0)
check("TSP 6 cities", "regret-2 from 1", L, 19)
check_vec("TSP 6 cities", "  tour", tour, e69, 6)
L, tour = pilot_nn(6, d, 0)
check("TSP 6 cities", "pilot NN from 1", L, 20)
check_vec("TSP 6 cities", "  tour", tour, e70, 6)

d = [[0, 7, 7, 9, 4, 9], [7, 0, 2, 5, 6, 4], [7, 2, 0, 3, 5, 2], [9, 5, 3, 0, 5, 1], [4, 6, 5, 5, 0, 6], [9, 4, 2, 1, 6, 0]]
L, tour = nearest_neighbour(6, d, 0)
check("EX_TC_1", "NN from 1", L, 25)
check_vec("EX_TC_1", "  tour", tour, e71, 6)
L, tour = nearest_neighbour(6, d, 1)
check("EX_TC_1", "NN from 2", L, 21)
check_vec("EX_TC_1", "  tour", tour, e72, 6)
L, tour = nearest_neighbour(6, d, 2)
check("EX_TC_1", "NN from 3", L, 23)
check_vec("EX_TC_1", "  tour", tour, e73, 6)
L, tour = nearest_neighbour(6, d, 3)
check("EX_TC_1", "NN from 4", L, 24)
check_vec("EX_TC_1", "  tour", tour, e74, 6)
L, tour = nearest_neighbour(6, d, 4)
check("EX_TC_1", "NN from 5", L, 21)
check_vec("EX_TC_1", "  tour", tour, e75, 6)
L, tour = nearest_neighbour(6, d, 5)
check("EX_TC_1", "NN from 6", L, 25)
check_vec("EX_TC_1", "  tour", tour, e76, 6)
L, tour = multi_start_nn(6, d)
check("EX_TC_1", "multi-start NN", L, 21)
check_vec("EX_TC_1", "  tour", tour, e77, 6)
L, tour = insertion(6, d, 0, 0)
check("EX_TC_1", "NI from 1", L, 21)
check_vec("EX_TC_1", "  tour", tour, e78, 6)
L, tour = insertion(6, d, 0, 1)
check("EX_TC_1", "FI from 1", L, 21)
check_vec("EX_TC_1", "  tour", tour, e79, 6)
L, tour = insertion(6, d, 0, 2)
check("EX_TC_1", "CI from 1", L, 21)
check_vec("EX_TC_1", "  tour", tour, e80, 6)

d = [[0, 3, 4, 4, 2, 3], [3, 0, 7, 6, 4, 2], [4, 7, 0, 4, 4, 6], [4, 6, 4, 0, 6, 4], [2, 4, 4, 6, 0, 5], [3, 2, 6, 4, 5, 0]]
L, tour = nearest_neighbour(6, d, 0)
check("EX_TC_2", "NN from 1", L, 20)
check_vec("EX_TC_2", "  tour", tour, e81, 6)
L, tour = nearest_neighbour(6, d, 1)
check("EX_TC_2", "NN from 2", L, 21)
check_vec("EX_TC_2", "  tour", tour, e82, 6)
L, tour = nearest_neighbour(6, d, 2)
check("EX_TC_2", "NN from 3", L, 20)
check_vec("EX_TC_2", "  tour", tour, e83, 6)
L, tour = nearest_neighbour(6, d, 3)
check("EX_TC_2", "NN from 4", L, 22)
check_vec("EX_TC_2", "  tour", tour, e84, 6)
L, tour = nearest_neighbour(6, d, 4)
check("EX_TC_2", "NN from 5", L, 19)
check_vec("EX_TC_2", "  tour", tour, e85, 6)
L, tour = nearest_neighbour(6, d, 5)
check("EX_TC_2", "NN from 6", L, 19)
check_vec("EX_TC_2", "  tour", tour, e86, 6)
L, tour = multi_start_nn(6, d)
check("EX_TC_2", "multi-start NN", L, 19)
check_vec("EX_TC_2", "  tour", tour, e87, 6)
L, tour = insertion(6, d, 0, 0)
check("EX_TC_2", "NI from 1", L, 21)
check_vec("EX_TC_2", "  tour", tour, e88, 6)
L, tour = insertion(6, d, 0, 1)
check("EX_TC_2", "FI from 1", L, 19)
check_vec("EX_TC_2", "  tour", tour, e89, 6)
L, tour = insertion(6, d, 0, 2)
check("EX_TC_2", "CI from 1", L, 20)
check_vec("EX_TC_2", "  tour", tour, e90, 6)

d = [[0, 3, 4, 3, 6], [3, 0, 4, 3, 5], [4, 4, 0, 1, 3], [3, 3, 1, 0, 4], [6, 5, 3, 4, 0]]
L, tour = nearest_neighbour(5, d, 0)
check("EX_TC_3", "NN from 1", L, 16)
check_vec("EX_TC_3", "  tour", tour, e91, 5)
L, tour = nearest_neighbour(5, d, 1)
check("EX_TC_3", "NN from 2", L, 15)
check_vec("EX_TC_3", "  tour", tour, e92, 5)
L, tour = nearest_neighbour(5, d, 2)
check("EX_TC_3", "NN from 3", L, 15)
check_vec("EX_TC_3", "  tour", tour, e93, 5)
L, tour = nearest_neighbour(5, d, 3)
check("EX_TC_3", "NN from 4", L, 15)
check_vec("EX_TC_3", "  tour", tour, e94, 5)
L, tour = nearest_neighbour(5, d, 4)
check("EX_TC_3", "NN from 5", L, 15)
check_vec("EX_TC_3", "  tour", tour, e95, 5)
L, tour = multi_start_nn(5, d)
check("EX_TC_3", "multi-start NN", L, 15)
check_vec("EX_TC_3", "  tour", tour, e96, 5)
L, tour = insertion(5, d, 0, 0)
check("EX_TC_3", "NI from 1", L, 16)
check_vec("EX_TC_3", "  tour", tour, e97, 5)
L, tour = insertion(5, d, 0, 1)
check("EX_TC_3", "FI from 1", L, 15)
check_vec("EX_TC_3", "  tour", tour, e98, 5)
L, tour = insertion(5, d, 0, 2)
check("EX_TC_3", "CI from 1", L, 16)
check_vec("EX_TC_3", "  tour", tour, e99, 5)
seed = 2026
L, tour, seed = randomized_nn(5, d, 0, 1, 2, seed)
check("EX_TC_3", "RCL run 1", L, 18)
check_vec("EX_TC_3", "  tour", tour, e100, 5)
L, tour, seed = randomized_nn(5, d, 0, 1, 2, seed)
check("EX_TC_3", "RCL run 2", L, 15)
check_vec("EX_TC_3", "  tour", tour, e101, 5)

print("\n%d checks, %d errors" % (checks, errors))

KP reference       greedy profit        17 OK
KP reference         x                2 2 1 1 OK
KP reference       greedy weight        10 OK
KP reference         x                1 1 2 2 OK
KP reference       greedy efficiency    16 OK
KP reference         x                2 1 2 1 OK
EX_KO_1            greedy profit        15 OK
EX_KO_1              x                1 2 1 1 1 OK
EX_KO_1            greedy weight        12 OK
EX_KO_1              x                2 1 1 2 1 OK
EX_KO_1            greedy efficiency    19 OK
EX_KO_1              x                1 1 2 2 1 OK
EX_KO_2            greedy profit        16 OK
EX_KO_2              x                2 1 1 1 2 OK
EX_KO_2            greedy weight        17 OK
EX_KO_2              x                1 2 1 2 2 OK
EX_KO_2            greedy efficiency    13 OK
EX_KO_2              x                2 2 1 1 1 OK
EX_KO_3            greedy profit        13 OK
EX_KO_3              x                1 2 1 1 1 OK
EX_KO_3            greedy weight    

## Try it yourself

Change the data in the cell below and run it again.

In [17]:
# change the data below and run the cell again
w = [5, 4, 3, 3, 2]                      # weights of the items (KP and BPP)
p = [10, 7, 6, 4, 3]                     # profits of the items (KP)
c = 9                                    # capacity of the knapsack and of the bins
t = [5, 5, 4, 4, 3, 3, 3]                # processing times of the jobs
m = 3                                    # number of machines
d = [[0, 5, 2, 2, 9, 6], [5, 0, 4, 3, 4, 3], [2, 4, 0, 4, 7, 5],
     [2, 3, 4, 0, 7, 4], [9, 4, 7, 7, 0, 2], [6, 3, 5, 4, 2, 0]]   # distances


def show(a):                             # items, bins, machines, cities numbered from 1
    return " ".join(str(v + 1) for v in a)


n = len(w)
z, x = greedy_order_kp(n, c, w, p, kp_order(n, p, w, 2))
print("KP greedy by efficiency: profit %d, selected items %s" % (z, show([i for i in range(n) if x[i] == 1])))
nb, bin_ = first_fit_decreasing(n, c, w)
print("BPP first fit decreasing: %d bins, bin of each item %s" % (nb, show(bin_)))
nb, bin_ = best_fit_decreasing(n, c, w)
print("BPP best fit decreasing:  %d bins, bin of each item %s" % (nb, show(bin_)))
cmax, mach = lpt(len(t), m, t)
print("P||Cmax LPT: makespan %d, machine of each job %s" % (cmax, show(mach)))
nc = len(d)
for name, (L, tour) in [("nearest neighbour", nearest_neighbour(nc, d, 0)),
                        ("multi-start NN", multi_start_nn(nc, d)),
                        ("farthest insertion", insertion(nc, d, 0, 1)),
                        ("cheapest insertion", insertion(nc, d, 0, 2)),
                        ("regret-2 insertion", regret2_insertion(nc, d, 0)),
                        ("pilot NN", pilot_nn(nc, d, 0))]:
    print("TSP %-18s length %3d, tour %s" % (name, L, show(tour)))
L, tour, seed = randomized_nn(nc, d, 0, 3, 10, 1)
print("TSP %-18s length %3d, tour %s" % ("randomized NN", L, show(tour)))

KP greedy by efficiency: profit 16, selected items 1 3
BPP first fit decreasing: 2 bins, bin of each item 1 1 2 2 2
BPP best fit decreasing:  2 bins, bin of each item 1 1 2 2 2
P||Cmax LPT: makespan 11, machine of each job 1 2 3 3 1 2 1
TSP nearest neighbour  length  24, tour 1 3 2 4 6 5
TSP multi-start NN     length  18, tour 2 4 1 3 6 5
TSP farthest insertion length  18, tour 1 4 2 5 6 3
TSP cheapest insertion length  18, tour 1 4 6 5 2 3
TSP regret-2 insertion length  18, tour 1 4 2 5 6 3
TSP pilot NN           length  18, tour 1 4 6 5 2 3
TSP randomized NN      length  20, tour 1 3 2 6 5 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).